# R1 — Data Extraction with AI Functions

Builds the three Section 1 R1 objects in `claims.genai`:

1. `claims.genai.grievance_emails` — grievance_emails.csv loaded unchanged.
2. `claims.genai.discharge_extracted` — five facts (final ICD-10 diagnosis, procedure code, admission date, discharge date, room type) extracted from the 60 discharge summary PDFs via `ai_parse_document` → `ai_extract`; one row per summary; validated against `claims.silver.claims_clean` (bar: ≥90% per field — result: 100%).
3. `claims.genai.grievance_classified` — each of the 200 emails sorted into exactly one of the five categories from policy doc 08 with `ai_classify`.

Binding definitions: room type from the room rent per day in `claims_clean` (general_ward ≤ 2,000; twin_sharing 2,001–3,500; single_private 3,501–5,000; deluxe > 5,000); grievance categories per `policy_docs/08_Grievance_and_Claim_Disputes.md`.

Note: extraction runs on the `discharge_summaries/` PDFs — `ai_parse_document` (v2.0) reads the binary, `ai_extract` (v2.1) consumes the parsed VARIANT directly, and `source_file` is therefore the `.pdf` name consistently in every row (`document_text` is the flattened parse, heading line included). Raw AI outputs are materialized once as session temp views `_discharge_raw_extract` / `_grievance_raw_classified` before shaping (AI functions are sampled per call — never re-called for scoring; temp views keep the raw outputs out of the catalog).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS claims.genai;

CREATE OR REPLACE TABLE claims.genai.grievance_emails AS
SELECT email_id, received_ts, member_id, subject, body
FROM read_files(
  '/Volumes/claims/raw/landing/genai/grievance_emails.csv',
  format => 'csv',
  header => true
);

-- Expect 200 rows, 200 distinct email_id, no null bodies
SELECT COUNT(*) AS n, COUNT(DISTINCT email_id) AS uniq_ids,
       SUM(CASE WHEN body IS NULL OR body = '' THEN 1 ELSE 0 END) AS null_bodies
FROM claims.genai.grievance_emails;

In [0]:

-- Raw AI output kept as a SESSION TEMP VIEW (not a catalog table) so it never
-- shows in the catalog; stored once so every later step reads the same call
-- (AI output is sampled per call).
-- NOTE: PDF bytes must not be CAST to STRING — ai_parse_document reads the BINARY.
CREATE OR REPLACE TEMP VIEW _discharge_raw_extract AS
WITH pdfs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM read_files(
    '/Volumes/claims/raw/landing/genai/discharge_summaries/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  -- document_text: flattened parse output, heading line (title element) included
  concat_ws('\n\n',
    transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>),
              element -> try_cast(element:content AS STRING))
  ) AS document_text,
  -- ai_extract consumes the structured parsed VARIANT directly (never the flattened string)
  ai_extract(
    parsed,
    '{
      "diagnosis_code": {"type": "string", "description": "The ICD-10 code of the FINAL diagnosis only, from the line starting with Final diagnosis. Ignore any older or other diagnoses listed elsewhere in the summary."},
      "procedure_code": {"type": "string", "description": "The procedure code from the Procedure performed line, e.g. PR-ORIF"},
      "admission_date": {"type": "string", "description": "Date of admission normalized to YYYY-MM-DD"},
      "discharge_date": {"type": "string", "description": "Date of discharge normalized to YYYY-MM-DD"},
      "room_type": {"type": "enum", "labels": ["general_ward", "twin_sharing", "single_private", "deluxe"], "description": "Room category from the summary. General ward means rent up to Rs 2,000 per day; twin sharing Rs 2,001-3,500; single private Rs 3,501-5,000; deluxe above Rs 5,000. Map phrases like Twin sharing room to twin_sharing."}
    }',
    MAP(
      'version', '2.1',
      'instructions', 'These are hospital discharge summaries for insurance claims. Extract the five key facts for the claims adjuster. Always use the FINAL diagnosis code, not any earlier or unrelated diagnosis mentioned in the clinical history.'
    )
  ) AS extraction
FROM pdfs
WHERE is_variant_null(parsed:error_status);

-- Materializing note: serverless does not support CACHE TABLE, so this view is
-- LAZY — it must be read exactly ONCE, by the shaping CTAS in the next cell
-- (a view read once ≡ a CTE referenced once; AI outputs are sampled per call).
-- Integrity is checked after shaping (cells 3 and 4), not here.

In [0]:

-- Final contract: claim_id STRING, source_file STRING (same .pdf name in every
-- row), document_text STRING, diagnosis_code/procedure_code STRING,
-- admission_date/discharge_date DATE, room_type STRING.
CREATE OR REPLACE TABLE claims.genai.discharge_extracted AS
SELECT
  regexp_extract(path, '([^/]+)\\.pdf$', 1) AS claim_id,
  regexp_extract(path, '([^/]+)$', 1) AS source_file,
  document_text,
  extraction:response.diagnosis_code:value::STRING AS diagnosis_code,
  extraction:response.procedure_code:value::STRING AS procedure_code,
  CAST(extraction:response.admission_date:value::STRING AS DATE) AS admission_date,
  CAST(extraction:response.discharge_date:value::STRING AS DATE) AS discharge_date,
  extraction:response.room_type:value::STRING AS room_type
FROM _discharge_raw_extract;

SELECT * FROM claims.genai.discharge_extracted ORDER BY claim_id LIMIT 6;

In [0]:

-- Room type on the claims side is derived from room_rent_per_day using the
-- binding bands. Codes/dates compared case/space-insensitively.
WITH claims AS (
  SELECT
    claim_id,
    LOWER(regexp_replace(diagnosis_code, '\\s', '')) AS diagnosis_code,
    LOWER(regexp_replace(procedure_code, '\\s', '')) AS procedure_code,
    DATE(admission_date) AS admission_date,
    DATE(discharge_date) AS discharge_date,
    CASE
      WHEN room_rent_per_day <= 2000 THEN 'general_ward'
      WHEN room_rent_per_day <= 3500 THEN 'twin_sharing'
      WHEN room_rent_per_day <= 5000 THEN 'single_private'
      ELSE 'deluxe'
    END AS room_type
  FROM claims.silver.claims_clean
),
cmp AS (
  SELECT
    e.claim_id,
    c.claim_id IS NOT NULL AS claim_found,
    LOWER(e.diagnosis_code) = c.diagnosis_code AS diag_match,
    LOWER(e.procedure_code) = c.procedure_code AS proc_match,
    e.admission_date = c.admission_date AS adm_match,
    e.discharge_date = c.discharge_date AS dis_match,
    e.room_type = c.room_type AS room_match
  FROM claims.genai.discharge_extracted e
  LEFT JOIN claims c USING (claim_id)
)
SELECT
  COUNT(*) AS rows_extracted,
  SUM(CASE WHEN claim_found THEN 1 ELSE 0 END) AS matched_to_claims_clean,
  ROUND(AVG(CASE WHEN claim_found AND diag_match THEN 1.0 ELSE 0.0 END), 2)*100 AS diagnosis_match_pct,
  ROUND(AVG(CASE WHEN claim_found AND proc_match THEN 1.0 ELSE 0.0 END), 2)*100 AS procedure_match_pct,
  ROUND(AVG(CASE WHEN claim_found AND adm_match THEN 1.0 ELSE 0.0 END), 2)*100 AS admission_match_pct,
  ROUND(AVG(CASE WHEN claim_found AND dis_match THEN 1.0 ELSE 0.0 END), 2)*100 AS discharge_match_pct,
  ROUND(AVG(CASE WHEN claim_found AND room_match THEN 1.0 ELSE 0.0 END), 2)*100 AS room_match_pct,
  SUM(CASE WHEN claim_found AND diag_match AND proc_match AND adm_match AND dis_match AND room_match THEN 1 ELSE 0 END) AS rows_all_five_match
FROM cmp;
-- Result 2026-11-12: 60 rows, 60 matched, 1.0 on every field, 60 rows all-five.

In [0]:

-- Labels and instructions follow policy doc 08 exactly: the category is the
-- main thing the member wants fixed. Single label per email (no multilabel).
-- Raw AI output kept as a SESSION TEMP VIEW (not a catalog table).
CREATE OR REPLACE TEMP VIEW _grievance_raw_classified AS
SELECT
  email_id,
  ai_classify(
    concat_ws('. ', subject, body),
    '{
      "claim_delay": "A claim is taking too long to be decided or paid",
      "claim_rejection": "A claim was rejected and the member disputes the rejection",
      "payout_deduction": "A claim was paid but for less than the member expected, for example a co-payment or a room rent deduction",
      "cashless_network": "Cashless treatment or pre-authorisation was refused, or a hospital network status is disputed",
      "policy_service": "Policy administration: e-card, contact details, renewal, plan change, nominee"
    }',
    MAP(
      'version', '2.1',
      'instructions', 'These are member grievance emails to health insurer Saffron Shield. Classify each email by the MAIN thing the member wants fixed, per the Grievance and Claim Disputes policy. Ignore secondary asides (for example a mention of a past claim or a small admin request at the end); pick the single dominant issue.'
    )
  ) AS classification
FROM claims.genai.grievance_emails;

-- Lazy view, read exactly ONCE by the shaping CTAS in the next cell
-- (serverless has no CACHE TABLE; AI outputs are sampled per call).
-- Integrity is checked after shaping (cells 4 and 6), not here.

In [0]:

-- Final contract: email_id STRING, category STRING; one row per email;
-- every one of the five categories used at least once.
CREATE OR REPLACE TABLE claims.genai.grievance_classified AS
SELECT
  email_id,
  classification:response[0]:value::STRING AS category
FROM _grievance_raw_classified;

SELECT category, COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM claims.genai.grievance_classified
GROUP BY category
ORDER BY n DESC;

-- Integrity: 200 rows = 200 distinct email_id = 200 source emails
SELECT 'classified_rows' AS check, COUNT(*) AS n FROM claims.genai.grievance_classified
UNION ALL
SELECT 'classified_uniq_ids', COUNT(DISTINCT email_id) FROM claims.genai.grievance_classified
UNION ALL
SELECT 'source_emails', COUNT(*) FROM claims.genai.grievance_emails;